# Quickstart: `corr_vars` on the Charité demo dataset

A hands-on notebook to run top to bottom (**Run → Run All Cells**). It builds a small ICU
cohort with `corr_vars`, applies inclusion criteria, adds clinical variables over a time
window, and ends with a Table 1 and two plots.

Two public services stand behind it, and neither needs an account:

- **Data**: `charite_mockdata`, a *fully synthetic* ICU dataset (20'000 ICU stays,
  2019–2025) in the table layout of the Charité data warehouse, served as Parquet files at
  <https://demo-data.corr.icu/v1> and read on demand over HTTPS with DuckDB.
- **Variable definitions** (*concepts*): the demo Concepts API at
  <https://demo-concepts.corr.icu> (project `demo`, public key `cak_demo_0d2b53ac`).

> **Synthetic data only.** The dataset contains no real patient data and is not
> clinically valid. The numbers below demonstrate the workflow; they are not findings.

**Runtime.** Expect a few minutes for the whole notebook, depending on your connection.
Each run fetches what its queries need from the dataset (on the order of 100 MB) and
processes all ~20'000 ICU stays, so give the variable cells a little time. Nothing is
stored permanently except where you ask for it (section 6).

**Requirements.** `corr_vars` with the `charite-demo` extra and Jupyter; see
[`README.md`](README.md) in this folder for one-line `uv` commands.

## 1. Setup

Imports, and the public demo endpoints and key.

In [ ]:
import os

import matplotlib.pyplot as plt
import polars as pl

from corr_vars import Cohort
from corr_vars.sources.charite_demo import NativeStatic

# Public demo services. Optional: the env vars point to a mirror or a local copy.
DATA_URL = os.environ.get("CORR_DEMO_DATA_URL", "https://demo-data.corr.icu/v1")
CONCEPTS_API_URL = os.environ.get("CORR_DEMO_CONCEPTS_URL", "https://demo-concepts.corr.icu/api")
PROJECT, API_KEY = "demo", "cak_demo_0d2b53ac"  # public, read-only demo key

## 2. Build a cohort

A `Cohort` needs an **observation level** (here one row per ICU stay), a **data source**
with its options, and the Concepts API project and key. The source's `path` is the public
dataset URL (`{"charite_demo": {}}` alone would use it too).

Building the cohort reads only the small administrative tables (a few MB). Clinical data
are fetched later, variable by variable.

In [ ]:
cohort = Cohort(
    obs_level="icu_stay",  # one row per ICU stay
    sources={"charite_demo": {"path": DATA_URL}},
    project=PROJECT,
    api_key=API_KEY,
    concepts_api_url=CONCEPTS_API_URL,
    load_default_vars=False,  # start from the bare cohort frame
    logger_args={"level": "WARNING"},  # quiet; use "INFO" to follow every extraction step
)

obs = cohort.obs
print(
    f"{len(cohort):,} ICU stays, {obs['case_id'].n_unique():,} hospital cases, "
    f"{obs['patient_id'].n_unique():,} patients"
)

`cohort.obs` is a [polars](https://pola.rs) DataFrame with one row per ICU stay:
identifiers, demographics (`age_on_admission`, `sex`), the time anchors used for time
windows (`hospital_admission`, `icu_admission`, `icu_discharge`, ...), the admission
diagnoses and `inhospital_death`.

In [ ]:
print(obs.columns)
obs.sort("icu_stay_id").head(5)

In [ ]:
obs.select(
    pl.col("age_on_admission").median().alias("median age"),
    (pl.col("sex") == "F").mean().round(3).alias("female (share)"),
    pl.col("inhospital_death").mean().round(3).alias("in-hospital death (share)"),
)

## 3. Inclusion criteria and the flowchart

Each step is recorded by the cohort's change tracker, which draws the study flowchart:

- `cohort.include_list([...])` for criteria on a variable;
- `with cohort.change_tracker("description") as track: track.filter(<polars expression>)`
  for anything else.

Criteria: adults, the **first ICU stay of each hospital case**, and an ICU stay of **at
least 24 hours** (so that first-24-hour features are defined).

In [ ]:
cohort.include_list(
    [
        {
            "variable": "age_on_admission",
            "operation": ">= 18",
            "label": "Adults",
            "operations_done": "adults (age >= 18 years)",
        },
    ]
)

# Number the ICU stays within each hospital case by admission time
cohort.obs = cohort.obs.with_columns(
    icu_stay_number=pl.col("icu_admission").rank("ordinal").over("case_id")
)
with cohort.change_tracker("first ICU stay of each hospital case") as track:
    track.filter(pl.col("icu_stay_number") == 1)

with cohort.change_tracker("ICU stays of at least 24 h") as track:
    track.filter(pl.col("icu_discharge") - pl.col("icu_admission") >= pl.duration(hours=24))

print(f"Study cohort: {len(cohort):,} ICU stays")
cohort.to_figureone(
    layout="profile",
    labels={
        "adults": "aged < 18 years",
        "first ICU stay": "not the first ICU stay of the hospital case",
        "at least 24 h": "ICU stay < 24 h",
    },
)

The source already keeps adults only by default, so the first step removes nobody; it is
listed because a protocol states it. The cell above shows the trial-profile layout
(screened, excluded with reasons, included); `cohort.figureone` gives the step-by-step
cascade instead. Both are matplotlib figures; save the profile with
`cohort.to_figureone(layout="profile", output_file="flowchart.pdf")`.

## 4. Add variables

A *concept* is a curated, versioned variable definition. `cohort.add_variable(name)`
fetches the definition from the Concepts API, runs it against the data, and stores the
result:

- **static** concepts (one value per stay) become columns of `cohort.obs`;
- **dynamic** concepts (time series) land in `cohort.obsm[name]`, one row per measurement
  or interval (`recordtime`, `recordtime_end`, `value`, `value_unit`, `attributes`).

`age_on_admission` is already a column of `cohort.obs`. Let's add a comorbidity score and
five time series. Derived concepts such as `pf_ratio` (PaO₂/FiO₂) and `sofa_score_total`
fetch the concepts they depend on by themselves. **This cell is the slow one** (it scans
lab, ventilation and medication data for all stays; roughly a minute).

In [ ]:
cohort.add_variable("elix_total_vw")  # Elixhauser comorbidity index (van Walraven), static

for name in [
    "pf_ratio",  # PaO2 / FiO2, mmHg
    "norepinephrine",  # infusion intervals
    "invasive_vent",  # ventilation episodes
    "aki_crea_based",  # KDIGO AKI stage from creatinine
    "sofa_score_total",  # daily SOFA score
]:
    cohort.add_variable(name)

print("static columns:", [c for c in cohort.obs.columns if c.startswith("elix")])
print("time series:   ", list(cohort.obsm))

In [ ]:
cohort.obsm["invasive_vent"].select(
    "icu_stay_id", "recordtime", "recordtime_end", "value", "value_unit"
).sort("icu_stay_id", "recordtime").head(5)

By default a dynamic variable covers the observation itself (here `icu_admission` to
`icu_discharge`). `add_variable` also takes an explicit window, `tmin` / `tmax`, given as a
column of `cohort.obs` or a `(column, offset)` tuple. For example, only the
norepinephrine given in the first 24 hours, stored under a new name:

In [ ]:
cohort.add_variable(
    "norepinephrine",
    save_as="norepinephrine_first_24h",
    tmin="icu_admission",
    tmax=("icu_admission", "+24h"),
)
print(
    f"{cohort.obsm['norepinephrine_first_24h']['icu_stay_id'].n_unique():,} stays with "
    "norepinephrine in the first 24 h"
)

### From time series to one value per stay

An analysis usually needs one value per stay: the worst P/F in the first 24 hours, whether
the patient was ventilated, the highest AKI stage. `NativeStatic` wraps a time series with
an **aggregation** (`select`: `!first`, `!last`, `!min`, `!max`, `!mean`, `!any`,
`!sum_interval`, ...) and a **time window** (`tmin`, `tmax`).

Always pass `tmin` and `tmax` to `NativeStatic`: without them the window is unbounded.

In [ ]:
first_24h = {"tmin": "icu_admission", "tmax": ("icu_admission", "+24h")}
whole_icu_stay = {"tmin": "icu_admission", "tmax": "icu_discharge"}

for feature in [
    NativeStatic("pf_worst_24h", select="!min value", base_var="pf_ratio", **first_24h),
    NativeStatic("norepinephrine_24h", select="!any", base_var="norepinephrine", **first_24h),
    NativeStatic("invasive_vent_24h", select="!any", base_var="invasive_vent", **first_24h),
    NativeStatic("sofa_first", select="!first value", base_var="sofa_score_total", **first_24h),
    NativeStatic("aki_crea_max", select="!max value", base_var="aki_crea_based", **whole_icu_stay),
    NativeStatic(
        "vent_hours_icu", select="!sum_interval value", base_var="invasive_vent", **whole_icu_stay
    ),
]:
    cohort.add_variable(feature)

cohort.obs.select(
    "pf_worst_24h",
    "norepinephrine_24h",
    "invasive_vent_24h",
    "sofa_first",
    "aki_crea_max",
    "vent_hours_icu",
).describe()

`aki_crea_max` is the highest KDIGO stage (0–3) during the ICU stay; `-1` marks
dialysis-dependent chronic kidney disease (not assessable).

Every definition that was used, including indirect dependencies, is recorded with its
version. This is what makes a cohort reproducible:

In [ ]:
pl.DataFrame(
    [
        {"concept": name, "version": r["version"], "status": r["status"]}
        for name, records in cohort.concept_versions.items()
        for r in records
    ]
)

## 5. Table 1 and plots

`cohort.to_tableone()` builds a [TableOne](https://tableone.readthedocs.io) from
`cohort.obs` or from a frame you pass. Here grouped by invasive ventilation in the first
24 hours:

In [ ]:
aki_labels = {-1: "Dialysis-dependent CKD", 0: "No AKI", 1: "Stage 1", 2: "Stage 2", 3: "Stage 3"}

table_df = cohort.obs.select(
    pl.when(pl.col("invasive_vent_24h"))
    .then(pl.lit("Ventilated in first 24 h"))
    .otherwise(pl.lit("Not ventilated"))
    .alias("group"),
    "age_on_admission",
    pl.col("sex").replace_strict({"F": "female", "M": "male"}),
    "elix_total_vw",
    "sofa_first",
    "pf_worst_24h",
    "norepinephrine_24h",
    "vent_hours_icu",
    pl.col("aki_crea_max").replace_strict(aki_labels, default=None, return_dtype=pl.Utf8),
    "inhospital_death",
)

table1 = cohort.to_tableone(
    df=table_df,
    groupby="group",
    overall=True,
    order={"aki_crea_max": list(aki_labels.values())},
    rename={
        "age_on_admission": "Age, years",
        "sex": "Sex",
        "elix_total_vw": "Elixhauser index (van Walraven)",
        "sofa_first": "SOFA, first 24 h",
        "pf_worst_24h": "Worst P/F, first 24 h, mmHg",
        "norepinephrine_24h": "Norepinephrine, first 24 h",
        "vent_hours_icu": "Invasive ventilation in ICU, h",
        "aki_crea_max": "Highest AKI stage (creatinine)",
        "inhospital_death": "In-hospital death",
    },
)
table1

P/F needs a charted FiO₂, which exists mainly for ventilated patients; hence the many
missing values in the non-ventilated group. Export the table with
`table1.to_excel("table1.xlsx")` or the helpers in `corr_vars.utils` (`tableone_to_latex`,
`tableone_to_markdown`, ...).

### Plot 1: P/F ratio over the first three days

Median and interquartile range in 6-hour bins, for patients ventilated within 24 hours,
by survival. `recordtime_relative` is the time in seconds since ICU admission.

In [ ]:
ventilated = cohort.obs.filter(pl.col("invasive_vent_24h")).select("icu_stay_id", "inhospital_death")

pf_course = (
    cohort.obsm["pf_ratio"]
    .join(ventilated, on="icu_stay_id")
    .filter(pl.col("recordtime_relative").is_between(0, 72 * 3600, closed="left"))
    .with_columns(hour=(pl.col("recordtime_relative") // (6 * 3600)) * 6 + 3)
    .group_by("inhospital_death", "hour")
    .agg(
        pl.col("value").median().alias("median"),
        pl.col("value").quantile(0.25).alias("q1"),
        pl.col("value").quantile(0.75).alias("q3"),
    )
    .drop_nulls("inhospital_death")
    .sort("hour")
)

fig, ax = plt.subplots(figsize=(7, 4))
for died, label, color in [(False, "Survivors", "#2a7ab9"), (True, "Non-survivors", "#d1495b")]:
    d = pf_course.filter(pl.col("inhospital_death") == died)
    ax.plot(d["hour"], d["median"], marker="o", color=color, label=label)
    ax.fill_between(d["hour"], d["q1"], d["q3"], color=color, alpha=0.15)
ax.set(
    xlabel="Hours since ICU admission",
    ylabel="P/F ratio (mmHg)",
    xticks=range(0, 73, 12),
    title="P/F ratio in patients ventilated within 24 h",
)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

### Plot 2: daily SOFA score by survival

`sofa_score_total` has one row per ICU day. The median score on days 1–7:

In [ ]:
sofa_by_day = (
    cohort.obsm["sofa_score_total"]
    .join(cohort.obs.select("icu_stay_id", "inhospital_death"), on="icu_stay_id")
    .drop_nulls("inhospital_death")
    .with_columns(day=(pl.col("recordtime_relative") // (24 * 3600) + 1).cast(pl.Int64))
    .filter(pl.col("day").is_between(1, 7))
    .group_by("inhospital_death", "day")
    .agg(pl.col("value").median().alias("median"), pl.len().alias("n"))
    .sort("day")
)

fig, ax = plt.subplots(figsize=(7, 4))
for died, label, color in [(False, "Survivors", "#2a7ab9"), (True, "Non-survivors", "#d1495b")]:
    d = sofa_by_day.filter(pl.col("inhospital_death") == died)
    ax.plot(d["day"], d["median"], marker="o", color=color, label=label)
ax.set(xlabel="ICU day", ylabel="Median SOFA score", title="Daily SOFA score", ylim=(0, None))
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 6. Optional: work from a local copy

Reading over HTTPS transfers only what each query touches, but every query pays network
round-trips. For repeated work, download the dataset once (about 195 MB; interrupted
downloads resume) and point the source at the folder. The variable definitions still come
from the Concepts API.

This section is **off by default**. Set `RUN_LOCAL_COPY = True` to try it.

In [ ]:
RUN_LOCAL_COPY = False  # set to True to download the dataset and rebuild the cohort from it
LOCAL_DIR = "~/data/charite_mockdata"  # where to keep the local copy

if RUN_LOCAL_COPY:
    from corr_vars.sources import charite_demo

    local_path = charite_demo.download(LOCAL_DIR, base_url=DATA_URL)  # skips files already there
    print("local copy:", local_path)

    local_cohort = Cohort(
        obs_level="icu_stay",
        sources={"charite_demo": {"path": local_path}},
        project=PROJECT,
        api_key=API_KEY,
        concepts_api_url=CONCEPTS_API_URL,
        load_default_vars=False,
        logger_args={"level": "WARNING"},
    )
    local_cohort.add_variable("sofa_score_total")
    print(
        f"{len(local_cohort):,} ICU stays, "
        f"{local_cohort.obsm['sofa_score_total'].height:,} daily SOFA scores (from the local copy)"
    )

## 7. Next steps

- Save your work: `cohort.save("my_study.corr3")` writes everything (including the concept
  versions) to one file; `Cohort.load(...)` reads it back offline. `cohort.to_parquet()`,
  `to_csv()` and `to_stata()` export `obs` and every `obsm` table.
- Read the [full demo tutorial](https://docs.corr-vars.de/tutorials/charite_demo.html): more
  concepts, surgical patients and windows before ICU admission, AKI by both KDIGO criteria,
  inspecting a definition's code and pinning versions.
- Browse all demo concepts, their documentation, code and version history in the
  [demo Concepts Browser](https://demo-concepts.corr.icu) (sign in as `demo`).
- The [documentation](https://docs.corr-vars.de): the `Cohort` API, aggregations and
  custom variables.